# Room 2.3 — Data Preprocessing

This notebook prepares the datasets used for the analysis of Room 2.3.

The preprocessing pipeline transforms the raw sensor telemetry into a consistent time-series representation suitable for exploratory analysis and subsequent modelling.

The available sensor data are processed independently and subsequently combined into a common 10-minute temporal resolution.

The final processed datasets produced in this notebook are used directly in the analysis of Room 2.3.

## 1. Indoor Air Quality (IAQ) Preprocessing

The IAQ sensor installed in Room 2.3 records the following variables:

- CO₂ concentration,
- temperature,
- relative humidity,
- light level,
- PIR-based motion detection.

The raw telemetry is converted into tabular form, temporally aligned, interpolated where necessary and resampled to common 10-minute intervals.

The resulting dataset is exported as:

`iaq_2_3.csv`

In [1]:
# 1.INDOOR AIR QUALITY PREPROCESSING — ROOM 2.3


import json

import numpy as np
import pandas as pd


json_file = "/content/iaq_2_3_raw.json"


with open(
    json_file,
    "r",
    encoding="utf-8"
) as file:
    data = json.load(file)


print("Telemetry keys:")
print(data.keys())


print("\nNumber of observations per variable:")

for key in data:

    print(
        f"{key}: {len(data[key])}"
    )

Telemetry keys:
dict_keys(['co2', 'temperature', 'humidity', 'light_level', 'pir'])

Number of observations per variable:
co2: 38906
temperature: 38895
humidity: 38895
light_level: 38895
pir: 38895


In [2]:
# 1.1 Convert individual telemetry variables to DataFrames

def telemetry_to_df(
    data,
    variable
):
    """
    Convert one telemetry variable from the raw JSON file
    into a timestamped pandas DataFrame.
    """

    df = pd.DataFrame(
        data[variable]
    )

    df["ts"] = pd.to_datetime(
        df["ts"],
        unit="ms"
    )

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    df = df.rename(
        columns={
            "value": variable
        }
    )

    return df

In [3]:
# Convert each available IAQ variable independently.

co2 = telemetry_to_df(
    data,
    "co2"
)

temperature = telemetry_to_df(
    data,
    "temperature"
)

humidity = telemetry_to_df(
    data,
    "humidity"
)

light = telemetry_to_df(
    data,
    "light_level"
)

pir = telemetry_to_df(
    data,
    "pir"
)


display(
    co2.head()
)

,ts,co2
0,2026-05-27 23:23:58.376,420
1,2026-05-27 23:13:58.448,420
2,2026-05-27 23:03:58.633,420
3,2026-05-27 22:53:58.818,420
4,2026-05-27 22:43:59.005,421


### Integration of Raw IAQ Variables

The individual telemetry variables do not necessarily share identical timestamps. They are therefore combined using outer joins, preserving all available observations before temporal interpolation and resampling.

In [4]:
# 1.2 Integrate raw IAQ variables


iaq = co2.copy()

iaq = iaq.merge(
    temperature,
    on="ts",
    how="outer"
)

iaq = iaq.merge(
    humidity,
    on="ts",
    how="outer"
)

iaq = iaq.merge(
    light,
    on="ts",
    how="outer"
)

iaq = iaq.merge(
    pir,
    on="ts",
    how="outer"
)


iaq = (
    iaq
    .sort_values("ts")
    .reset_index(drop=True)
)


print(
    "Raw integrated IAQ shape:",
    iaq.shape
)

display(
    iaq.head()
)

Raw integrated IAQ shape: (38906, 6)


,ts,co2,temperature,humidity,light_level,pir
0,2025-08-31 21:03:41.209,419,29.9,42.5,0.0,0.0
1,2025-08-31 21:13:41.046,419,29.8,42.5,0.0,0.0
2,2025-08-31 21:23:40.832,420,29.8,42.5,0.0,0.0
3,2025-08-31 21:33:40.694,420,29.8,42.0,0.0,0.0
4,2025-08-31 21:43:40.421,421,29.8,42.0,0.0,0.0


In [5]:
# 1.3 Initial data-quality assessment


print(
    "Shape:",
    iaq.shape
)

print("\nData types:")
print(
    iaq.dtypes
)

print("\nMissing values:")
print(
    iaq.isna().sum()
)

Shape: (38906, 6)

Data types:
ts             datetime64[ns]
co2                     int64
temperature           float64
humidity              float64
light_level           float64
pir                   float64
dtype: object

Missing values:
ts              0
co2             0
temperature    11
humidity       11
light_level    11
pir            11
dtype: int64


### Interpolation of Missing Sensor Values

Missing values in temperature, humidity, light level and PIR measurements are interpolated before temporal resampling.

This step preserves the original preprocessing logic used for the Room 2.3 dataset and reduces gaps caused by asynchronous sensor reporting.

In [6]:
# 1.4 Interpolate missing non-CO₂ measurements

iaq = (
    iaq
    .sort_values("ts")
    .reset_index(drop=True)
)


interpolation_columns = [
    "temperature",
    "humidity",
    "light_level",
    "pir"
]


iaq[
    interpolation_columns
] = (
    iaq[
        interpolation_columns
    ]
    .interpolate(
        method="linear",
        limit_direction="both"
    )
)


print(
    "Missing values after interpolation:"
)

print(
    iaq.isna().sum()
)

Missing values after interpolation:
ts             0
co2            0
temperature    0
humidity       0
light_level    0
pir            0
dtype: int64


### Temporal Resampling to 10-Minute Intervals

The combined IAQ measurements are resampled to a common 10-minute temporal resolution.

Mean values are calculated for the continuous environmental variables, while the maximum PIR value within each interval is retained to represent whether motion was detected during that period.

Remaining gaps are interpolated to obtain a complete 10-minute time series.

In [7]:
# 1.5 Resample IAQ data to 10-minute intervals

iaq_10min = (
    iaq
    .set_index("ts")
    .resample("10min")
    .agg({
        "co2": "mean",
        "temperature": "mean",
        "humidity": "mean",
        "light_level": "mean",
        "pir": "max"
    })
    .interpolate(
        limit_direction="both"
    )
    .reset_index()
)


print(
    "Shape after resampling:",
    iaq_10min.shape
)

print("\nTime range:")
print(
    iaq_10min["ts"].min(),
    "→",
    iaq_10min["ts"].max()
)

print("\nMissing values:")
print(
    iaq_10min.isna().sum()
)


display(
    iaq_10min.head()
)

Shape after resampling: (38751, 6)

Time range:
2025-08-31 21:00:00 → 2026-05-27 23:20:00

Missing values:
ts             0
co2            0
temperature    0
humidity       0
light_level    0
pir            0
dtype: int64


,ts,co2,temperature,humidity,light_level,pir
0,2025-08-31 21:00:00,419.0,29.9,42.5,0.0,0.0
1,2025-08-31 21:10:00,419.0,29.8,42.5,0.0,0.0
2,2025-08-31 21:20:00,420.0,29.8,42.5,0.0,0.0
3,2025-08-31 21:30:00,420.0,29.8,42.0,0.0,0.0
4,2025-08-31 21:40:00,421.0,29.8,42.0,0.0,0.0


In [8]:
# 1.6 Final IAQ validation

print(
    "Final IAQ shape:",
    iaq_10min.shape
)

print("\nDuplicate timestamps:")
print(
    iaq_10min[
        "ts"
    ]
    .duplicated()
    .sum()
)

print("\nSummary statistics:")

display(
    iaq_10min.describe()
)

Final IAQ shape: (38751, 6)

Duplicate timestamps:
0

Summary statistics:


,ts,co2,temperature,humidity,light_level,pir
count,38751,38751.000000,38751.000000,38751.000000,38751.000000,38751.000000
mean,2026-01-13 10:10:00,488.763903,22.361406,44.835143,0.720562,0.039586
min,2025-08-31 21:00:00,377.000000,17.100000,19.500000,0.000000,0.000000
25%,2025-11-07 03:35:00,419.000000,20.100000,39.500000,0.000000,0.000000
50%,2026-01-13 10:10:00,442.000000,21.800000,44.500000,0.000000,0.000000
75%,2026-03-21 16:45:00,490.000000,24.100000,50.000000,1.000000,0.000000
max,2026-05-27 23:20:00,2317.000000,29.900000,70.000000,3.000000,1.000000
std,NaN,144.607443,2.785544,7.309566,0.917499,0.194722


In [9]:
# 1.7 Export cleaned IAQ dataset


output_file = (
    "iaq_2_3.csv"
)


iaq_10min.to_csv(
    output_file,
    index=False
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    iaq_10min.shape
)

Saved: iaq_2_3.csv
Final shape: (38751, 6)


## 2. Workplace Occupancy Preprocessing

Room 2.3 contains a Workplace Occupancy sensor that provides information about the number of people detected in the room.

The available telemetry includes:

- `people_count_all`,
- `people_count_max`,
- `region_count`.

The raw telemetry variables are first converted into separate time series and then merged using their timestamps. The resulting dataset is aligned to 10-minute intervals.

The latest available occupancy state is retained within each interval, while forward filling is used to propagate the most recently known occupancy information where necessary.

The cleaned dataset is exported as:

`workplace_occupancy_2_3_clean.csv`

In [10]:
# 2. WORKPLACE OCCUPANCY PREPROCESSING — ROOM 2.3


json_file = "/content/workplace_occupancy_2_3_raw.json"

with open(
    json_file,
    "r",
    encoding="utf-8"
) as file:
    data = json.load(file)


print("Telemetry keys:")
print(data.keys())


print("\nNumber of observations per variable:")

for key in data:
    print(
        f"{key}: {len(data[key])}"
    )

Telemetry keys:
dict_keys(['people_count_all', 'people_count_max', 'region_count'])

Number of observations per variable:
people_count_all: 28648
people_count_max: 7334
region_count: 28648


In [11]:
# 2.1 Convert telemetry variables to DataFrames


def telemetry_to_df(
    data,
    variable
):
    """
    Convert one Workplace Occupancy telemetry variable
    into a timestamped pandas DataFrame.
    """

    df = pd.DataFrame(
        data[variable]
    )

    df["ts"] = pd.to_datetime(
        df["ts"],
        unit="ms"
    )

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    df = df.rename(
        columns={
            "value": variable
        }
    )

    return df

In [12]:
people_all = telemetry_to_df(
    data,
    "people_count_all"
)

people_max = telemetry_to_df(
    data,
    "people_count_max"
)

region = telemetry_to_df(
    data,
    "region_count"
)

### Integration of Workplace Occupancy Variables

The three Workplace Occupancy telemetry variables are combined using their timestamps. Outer joins are used in order to preserve all available measurements before temporal resampling.

In [13]:
# 2.2 Integrate Workplace Occupancy variables


people = people_all.copy()

people = people.merge(
    people_max,
    on="ts",
    how="outer"
)

people = people.merge(
    region,
    on="ts",
    how="outer"
)


people = (
    people
    .sort_values("ts")
    .reset_index(drop=True)
)


print(
    "Integrated Workplace Occupancy shape:",
    people.shape
)

display(
    people.head()
)

Integrated Workplace Occupancy shape: (28651, 4)


,ts,people_count_all,people_count_max,region_count
0,2025-08-31 22:59:45.507,0.0,0.0,0.0
1,2025-08-31 23:11:20.761,0.0,0.0,0.0
2,2025-08-31 23:59:45.595,0.0,0.0,0.0
3,2025-09-01 00:59:45.711,0.0,0.0,0.0
4,2025-09-01 01:59:45.822,0.0,0.0,0.0


In [14]:
# 2.3 Initial data-quality assessment


print(
    "Shape:",
    people.shape
)

print("\nData types:")
print(
    people.dtypes
)

print("\nMissing values:")
print(
    people.isna().sum()
)

Shape: (28651, 4)

Data types:
ts                  datetime64[ns]
people_count_all           float64
people_count_max           float64
region_count               float64
dtype: object

Missing values:
ts                      0
people_count_all        3
people_count_max    21317
region_count            3
dtype: int64


### Temporal Resampling to 10-Minute Intervals

Before resampling, missing values of `people_count_max` are forward filled using the most recently available value.

The dataset is then resampled to 10-minute intervals. For each interval, the latest available measurement is retained and remaining gaps are forward filled to preserve the latest known occupancy state.

In [15]:
# 2.4 Prepare occupancy measurements for resampling

people = (
    people
    .sort_values("ts")
)


# Preserve the latest available people_count_max value.

people["people_count_max"] = (
    people[
        "people_count_max"
    ]
    .ffill()
)

In [16]:
# 2.5 Resample to 10-minute intervals


people = (
    people
    .set_index("ts")
    .resample("10min")
    .last()
    .ffill()
    .reset_index()
)


print(
    "Shape after resampling:",
    people.shape
)

print("\nTime range:")
print(
    people["ts"].min(),
    "→",
    people["ts"].max()
)

print("\nMissing values:")
print(
    people.isna().sum()
)


display(
    people.head()
)

Shape after resampling: (44341, 4)

Time range:
2025-08-31 22:50:00 → 2026-07-05 20:50:00

Missing values:
ts                  0
people_count_all    0
people_count_max    0
region_count        0
dtype: int64


,ts,people_count_all,people_count_max,region_count
0,2025-08-31 22:50:00,0.0,0.0,0.0
1,2025-08-31 23:00:00,0.0,0.0,0.0
2,2025-08-31 23:10:00,0.0,0.0,0.0
3,2025-08-31 23:20:00,0.0,0.0,0.0
4,2025-08-31 23:30:00,0.0,0.0,0.0


In [17]:
# 2.6 Final Workplace Occupancy validation


print(
    "Final shape:",
    people.shape
)

print("\nDuplicate timestamps:")
print(
    people[
        "ts"
    ]
    .duplicated()
    .sum()
)

print("\nSummary statistics:")

display(
    people.describe(
        include="all"
    )
)

Final shape: (44341, 4)

Duplicate timestamps:
0

Summary statistics:


,ts,people_count_all,people_count_max,region_count
count,44341,44341.00000,44341.000000,44341.0
mean,2026-02-01 21:50:00,0.70145,1.581629,0.0
min,2025-08-31 22:50:00,0.00000,0.000000,0.0
25%,2025-11-16 22:20:00,0.00000,0.000000,0.0
50%,2026-02-01 21:50:00,0.00000,0.000000,0.0
75%,2026-04-19 21:20:00,0.00000,1.000000,0.0
max,2026-07-05 20:50:00,40.00000,44.000000,0.0
std,NaN,3.07207,4.336063,0.0


In [18]:
# 2.7 Export cleaned Workplace Occupancy dataset


output_file = (
    "workplace_occupancy_2_3_clean.csv"
)


people.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    people.shape
)

Saved: workplace_occupancy_2_3_clean.csv
Final shape: (44341, 4)


## 3. Magnetic Contact Sensor Preprocessing

Room 2.3 contains five magnetic contact sensors used to monitor the state of the room windows.

Each sensor records the binary state of one monitored opening. The raw measurements are independently converted into timestamped time series and resampled to 10-minute intervals.

Because magnetic contact sensors report state changes rather than continuous measurements, the latest known state is propagated forward until a new observation becomes available.

The preprocessing procedure includes:

- loading the five raw Magnetic Contact JSON files,
- converting timestamps and sensor values,
- resampling each sensor independently to 10-minute intervals,
- combining the five sensor time series,
- restricting the dataset to the period where all sensors have become available,
- calculating the total number of open windows,
- calculating an additional dynamic window indicator based on the selected sensors used in the subsequent analysis.

The cleaned dataset is exported as:

`magnetic_contacts_2_3_clean.csv`

In [19]:
# 3. MAGNETIC CONTACT SENSOR PREPROCESSING — ROOM 2.3


import os

files_mc = {
    "mc_1": "/content/F2_2.3-MC-1_raw.json",
    "mc_2": "/content/F2_2.3-MC-2_raw.json",
    "mc_3": "/content/F2_2.3-MC-3_raw.json",
    "mc_4": "/content/F2_2.3-MC-4_raw.json",
    "mc_5": "/content/F2_2.3-MC-5_raw.json"
}


# Verify that all expected raw JSON files are available.

for name, path in files_mc.items():
    print(
        name,
        "->",
        os.path.exists(path),
        path
    )

mc_1 -> True /content/F2_2.3-MC-1_raw.json
mc_2 -> True /content/F2_2.3-MC-2_raw.json
mc_3 -> True /content/F2_2.3-MC-3_raw.json
mc_4 -> True /content/F2_2.3-MC-4_raw.json
mc_5 -> True /content/F2_2.3-MC-5_raw.json


In [20]:
# 3.1 Load individual Magnetic Contact sensors


def load_magnetic_contact(
    json_path,
    column_name
):
    """
    Load one Magnetic Contact JSON file and return
    a timestamped DataFrame containing its binary state.
    """

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as file:
        data = json.load(file)

    records = data.get(
        "magnet_status",
        []
    )

    df = pd.DataFrame(records)

    # Return a structured empty DataFrame if no data exist.
    if df.empty:
        return pd.DataFrame(
            columns=[
                "ts",
                column_name
            ]
        )

    df["ts"] = pd.to_datetime(
        df["ts"],
        unit="ms"
    )

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    df = df.rename(
        columns={
            "value": column_name
        }
    )

    df = (
        df[
            [
                "ts",
                column_name
            ]
        ]
        .sort_values("ts")
        .reset_index(drop=True)
    )

    return df

In [21]:
mc_1 = load_magnetic_contact(
    files_mc["mc_1"],
    "mc_1"
)

mc_2 = load_magnetic_contact(
    files_mc["mc_2"],
    "mc_2"
)

mc_3 = load_magnetic_contact(
    files_mc["mc_3"],
    "mc_3"
)

mc_4 = load_magnetic_contact(
    files_mc["mc_4"],
    "mc_4"
)

mc_5 = load_magnetic_contact(
    files_mc["mc_5"],
    "mc_5"
)


for name, df in {
    "mc_1": mc_1,
    "mc_2": mc_2,
    "mc_3": mc_3,
    "mc_4": mc_4,
    "mc_5": mc_5
}.items():

    print(
        f"{name}: {len(df)} observations | "
        f"{df['ts'].min() if not df.empty else 'no data'} → "
        f"{df['ts'].max() if not df.empty else 'no data'}"
    )

mc_1: 2885 observations | 2025-08-31 23:05:25.902000 → 2026-07-05 19:59:51.208000
mc_2: 1845 observations | 2025-09-01 11:27:24.581000 → 2026-07-05 14:41:48.969000
mc_3: 4413 observations | 2025-09-01 07:34:23.354000 → 2026-07-05 14:52:54.413000
mc_4: 1106 observations | 2025-09-01 14:39:11.373000 → 2026-07-05 15:02:13.177000
mc_5: 1417 observations | 2025-09-01 14:46:17.613000 → 2026-07-05 15:07:37.357000


### Temporal Alignment to 10-Minute Intervals

Each Magnetic Contact time series is resampled independently to 10-minute intervals.

For every interval, the latest available sensor state is retained. Forward filling is then applied because the most recently observed window state remains valid until a new change is recorded.

In [22]:
# 3.2 Resample Magnetic Contact sensors


def resample_magnetic_contact(
    df,
    column_name
):
    """
    Resample one Magnetic Contact sensor to 10-minute
    intervals and propagate the latest known state.
    """

    if df.empty:
        return pd.DataFrame(
            columns=[
                "ts",
                column_name
            ]
        )

    result = (
        df
        .set_index("ts")
        .resample("10min")
        .last()
        .ffill()
        .reset_index()
    )

    return result

In [23]:
mc_1_10min = resample_magnetic_contact(
    mc_1,
    "mc_1"
)

mc_2_10min = resample_magnetic_contact(
    mc_2,
    "mc_2"
)

mc_3_10min = resample_magnetic_contact(
    mc_3,
    "mc_3"
)

mc_4_10min = resample_magnetic_contact(
    mc_4,
    "mc_4"
)

mc_5_10min = resample_magnetic_contact(
    mc_5,
    "mc_5"
)

In [24]:
# 3.3 Integrate the five Magnetic Contact sensors


magnetic_contacts = (
    mc_1_10min.copy()
)


for df in [
    mc_2_10min,
    mc_3_10min,
    mc_4_10min,
    mc_5_10min
]:

    magnetic_contacts = magnetic_contacts.merge(
        df,
        on="ts",
        how="outer"
    )


magnetic_contacts = (
    magnetic_contacts
    .sort_values("ts")
    .reset_index(drop=True)
)


sensor_columns = [
    "mc_1",
    "mc_2",
    "mc_3",
    "mc_4",
    "mc_5"
]


# Propagate the latest known state after integration.

magnetic_contacts[
    sensor_columns
] = (
    magnetic_contacts[
        sensor_columns
    ]
    .ffill()
)

In [25]:
# 3.4 Restrict dataset to common sensor availability


first_valid_times = {
    column:
        magnetic_contacts.loc[
            magnetic_contacts[
                column
            ].notna(),
            "ts"
        ].min()

    for column in sensor_columns
}


common_start = (
    max(
        first_valid_times.values()
    )
)


print(
    "First available measurement per sensor:"
)

print(
    first_valid_times
)

print(
    "\nCommon start:"
)

print(
    common_start
)


magnetic_contacts_clean = (
    magnetic_contacts[
        magnetic_contacts["ts"] >= common_start
    ]
    .copy()
    .reset_index(drop=True)
)

First available measurement per sensor:
{'mc_1': Timestamp('2025-08-31 23:00:00'), 'mc_2': Timestamp('2025-09-01 11:20:00'), 'mc_3': Timestamp('2025-09-01 07:30:00'), 'mc_4': Timestamp('2025-09-01 14:30:00'), 'mc_5': Timestamp('2025-09-01 14:40:00')}

Common start:
2025-09-01 14:40:00


### Room-Level Window Variables

Two aggregate window-state variables are calculated.

`open_windows_all` represents the total number of open windows across all five Magnetic Contact sensors.

`open_windows_dynamic` is calculated using sensors `mc_1`, `mc_2`, `mc_4` and `mc_5`, reproducing the dynamic window-state representation used in the subsequent Room 2.3 analysis.

In [26]:
# 3.5 Create aggregate window-state variables

magnetic_contacts_clean[
    "open_windows_all"
] = (
    magnetic_contacts_clean[
        sensor_columns
    ]
    .sum(axis=1)
    .astype(int)
)


dynamic_sensor_columns = [
    "mc_1",
    "mc_2",
    "mc_4",
    "mc_5"
]


magnetic_contacts_clean[
    "open_windows_dynamic"
] = (
    magnetic_contacts_clean[
        dynamic_sensor_columns
    ]
    .sum(axis=1)
    .astype(int)
)

In [27]:
# 3.6 Final Magnetic Contact validation

print(
    "Final shape:",
    magnetic_contacts_clean.shape
)

print("\nTime range:")
print(
    magnetic_contacts_clean["ts"].min(),
    "→",
    magnetic_contacts_clean["ts"].max()
)

print("\nMissing values:")
print(
    magnetic_contacts_clean.isna().sum()
)

print("\nDuplicate timestamps:")
print(
    magnetic_contacts_clean[
        "ts"
    ]
    .duplicated()
    .sum()
)

print("\nUnique values per sensor:")

for column in sensor_columns:
    print(
        column,
        sorted(
            magnetic_contacts_clean[
                column
            ]
            .dropna()
            .unique()
        )
    )


display(
    magnetic_contacts_clean.head()
)


display(
    magnetic_contacts_clean.describe()
)


print(
    "\nDistribution of open_windows_all:"
)

print(
    magnetic_contacts_clean[
        "open_windows_all"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nDistribution of open_windows_dynamic:"
)

print(
    magnetic_contacts_clean[
        "open_windows_dynamic"
    ]
    .value_counts()
    .sort_index()
)

Final shape: (44240, 8)

Time range:
2025-09-01 14:40:00 → 2026-07-05 19:50:00

Missing values:
ts                      0
mc_1                    0
mc_2                    0
mc_3                    0
mc_4                    0
mc_5                    0
open_windows_all        0
open_windows_dynamic    0
dtype: int64

Duplicate timestamps:
0

Unique values per sensor:
mc_1 [np.float64(0.0), np.float64(1.0)]
mc_2 [np.float64(0.0), np.float64(1.0)]
mc_3 [np.float64(0.0), np.float64(1.0)]
mc_4 [np.float64(0.0), np.float64(1.0)]
mc_5 [np.float64(0.0), np.float64(1.0)]


,ts,mc_1,mc_2,mc_3,mc_4,mc_5,open_windows_all,open_windows_dynamic
0,2025-09-01 14:40:00,1.0,0.0,0.0,0.0,0.0,1,1
1,2025-09-01 14:50:00,1.0,0.0,0.0,0.0,0.0,1,1
2,2025-09-01 15:00:00,1.0,0.0,0.0,0.0,0.0,1,1
3,2025-09-01 15:10:00,1.0,0.0,0.0,0.0,0.0,1,1
4,2025-09-01 15:20:00,1.0,0.0,0.0,0.0,0.0,1,1


,ts,mc_1,mc_2,mc_3,mc_4,mc_5,open_windows_all,open_windows_dynamic
count,44240,44240.000000,44240.000000,44240.000000,44240.000000,44240.000000,44240.000000,44240.000000
mean,2026-02-02 05:15:00,0.348395,0.074638,0.681397,0.077034,0.087658,1.269123,0.587726
min,2025-09-01 14:40:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-11-17 09:57:30,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
50%,2026-02-02 05:15:00,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,2026-04-20 00:32:30,1.000000,0.000000,1.000000,0.000000,0.000000,2.000000,1.000000
max,2026-07-05 19:50:00,1.000000,1.000000,1.000000,1.000000,1.000000,5.000000,4.000000
std,NaN,0.476467,0.262810,0.465940,0.266649,0.282800,0.943196,0.808356



Distribution of open_windows_all:
open_windows_all
0     8625
1    20726
2    10210
3     3813
4      769
5       97
Name: count, dtype: int64

Distribution of open_windows_dynamic:
open_windows_dynamic
0    25467
1    13162
2     4243
3     1119
4      249
Name: count, dtype: int64


In [28]:
# 3.7 Export cleaned Magnetic Contact dataset


magnetic_contacts_clean = (
    magnetic_contacts_clean
    .drop(
        columns=[
            "open_windows"
        ],
        errors="ignore"
    )
)


output_file = (
    "magnetic_contacts_2_3_clean.csv"
)


magnetic_contacts_clean.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    magnetic_contacts_clean.shape
)

Saved: magnetic_contacts_2_3_clean.csv
Final shape: (44240, 8)


## 4. Integration of Room 2.3 Sensor Data

The cleaned datasets from the three sensor sources are combined to construct the final integrated dataset for Room 2.3.

The integration combines:

- Indoor Air Quality measurements,
- Workplace Occupancy measurements,
- Magnetic Contact measurements.

The IAQ and Workplace Occupancy datasets are first merged using their common 10-minute timestamps. The aggregated window-state variable is then added from the Magnetic Contact dataset.

Observations without an available window-state measurement are removed, while selected discrete variables are converted to integer format.

The resulting dataset constitutes the final dataset used for the subsequent analysis of Room 2.3 and is exported as:

`room_2_3_final.csv`

In [29]:
# 4. SENSOR DATA INTEGRATION — ROOM 2.3


iaq = pd.read_csv(
    "/content/iaq_2_3.csv",
    parse_dates=["ts"]
)

wo = pd.read_csv(
    "/content/workplace_occupancy_2_3_clean.csv",
    parse_dates=["ts"]
)

mc = pd.read_csv(
    "/content/magnetic_contacts_2_3_clean.csv",
    parse_dates=["ts"]
)


print("IAQ:", iaq.shape)
print("Workplace Occupancy:", wo.shape)
print("Magnetic Contacts:", mc.shape)

IAQ: (38751, 6)
Workplace Occupancy: (44341, 4)
Magnetic Contacts: (44240, 8)


In [30]:
# 4.1 Inspect temporal coverage


datasets = {
    "IAQ": iaq,
    "Workplace Occupancy": wo,
    "Magnetic Contacts": mc
}


for name, df in datasets.items():

    print(
        f"{name}: "
        f"{df['ts'].min()} → {df['ts'].max()}"
    )

IAQ: 2025-08-31 21:00:00 → 2026-05-27 23:20:00
Workplace Occupancy: 2025-08-31 22:50:00 → 2026-07-05 20:50:00
Magnetic Contacts: 2025-09-01 14:40:00 → 2026-07-05 19:50:00


### Temporal Integration

The IAQ and Workplace Occupancy datasets are combined using an inner join, retaining timestamps available in both datasets.

The total number of open windows (`open_windows_all`) is subsequently added from the Magnetic Contact dataset using a left join. Observations without an available window-state measurement are excluded from the final dataset.

In [31]:
# 4.2 Integrate IAQ, Workplace Occupancy and window data


room_2_3 = (
    iaq
    .merge(
        wo,
        on="ts",
        how="inner"
    )
    .merge(
        mc[
            [
                "ts",
                "open_windows_all"
            ]
        ],
        on="ts",
        how="left"
    )
)


print(
    "Shape after integration:",
    room_2_3.shape
)

print(
    "\nMissing values before final restriction:"
)

print(
    room_2_3.isna().sum()
)

Shape after integration: (38740, 10)

Missing values before final restriction:
ts                   0
co2                  0
temperature          0
humidity             0
light_level          0
pir                  0
people_count_all     0
people_count_max     0
region_count         0
open_windows_all    95
dtype: int64


In [32]:
# 4.3 Retain observations with available window-state data

room_2_3_final = (
    room_2_3
    .dropna(
        subset=[
            "open_windows_all"
        ]
    )
    .copy()
    .reset_index(drop=True)
)


print(
    "Shape after window-state restriction:",
    room_2_3_final.shape
)

Shape after window-state restriction: (38645, 10)


In [33]:
# 4.4 Convert discrete variables to integer representation


integer_columns = [
    "pir",
    "people_count_all",
    "people_count_max",
    "region_count",
    "open_windows_all"
]


# Ensure that PIR remains binary.

room_2_3_final["pir"] = (
    room_2_3_final["pir"]
    .round()
    .clip(0, 1)
)


# Convert discrete variables to integers.

room_2_3_final[
    integer_columns
] = (
    room_2_3_final[
        integer_columns
    ]
    .round()
    .astype(int)
)

### Final Dataset Validation

Before export, the final Room 2.3 dataset is checked for temporal coverage, missing values, duplicate timestamps and valid PIR states.

The number of unique values per variable is also inspected to verify the final structure of the integrated dataset.

In [34]:
# 4.5 Final validation


print(
    "Final Room 2.3 dataset shape:",
    room_2_3_final.shape
)


print("\nTime range:")
print(
    room_2_3_final["ts"].min(),
    "→",
    room_2_3_final["ts"].max()
)


print("\nMissing values:")
print(
    room_2_3_final.isna().sum()
)


print("\nDuplicate timestamps:")
print(
    room_2_3_final[
        "ts"
    ]
    .duplicated()
    .sum()
)


print("\nUnique PIR values:")
print(
    sorted(
        room_2_3_final[
            "pir"
        ].unique()
    )
)


print("\nNumber of unique values per variable:")

for column in room_2_3_final.columns:

    if column != "ts":

        print(
            f"{column}: "
            f"{room_2_3_final[column].nunique()}"
        )


display(
    room_2_3_final.head()
)


display(
    room_2_3_final.describe(
        include="all"
    )
)

Final Room 2.3 dataset shape: (38645, 10)

Time range:
2025-09-01 14:40:00 → 2026-05-27 23:20:00

Missing values:
ts                  0
co2                 0
temperature         0
humidity            0
light_level         0
pir                 0
people_count_all    0
people_count_max    0
region_count        0
open_windows_all    0
dtype: int64

Duplicate timestamps:
0

Unique PIR values:
[np.int64(0), np.int64(1)]

Number of unique values per variable:
co2: 1129
temperature: 210
humidity: 166
light_level: 9
pir: 2
people_count_all: 37
people_count_max: 42
region_count: 1
open_windows_all: 6


,ts,co2,temperature,humidity,light_level,pir,people_count_all,people_count_max,region_count,open_windows_all
0,2025-09-01 14:40:00,544.0,29.9,37.5,1.0,0,0,2,0,1
1,2025-09-01 14:50:00,544.0,29.9,37.5,1.0,1,0,2,0,1
2,2025-09-01 15:00:00,543.0,29.9,37.5,1.0,0,0,2,0,1
3,2025-09-01 15:10:00,543.0,29.9,37.5,1.0,0,0,2,0,1
4,2025-09-01 15:20:00,544.0,29.9,37.5,1.0,0,0,2,0,1


,ts,co2,temperature,humidity,light_level,pir,people_count_all,people_count_max,region_count,open_windows_all
count,38645,38645.000000,38645.000000,38645.000000,38645.00000,38645.000000,38645.000000,38645.000000,38645.0,38645.000000
mean,2026-01-13 19:00:00,488.847794,22.340959,44.848392,0.72096,0.039410,0.762453,1.674499,0.0,1.251029
min,2025-09-01 14:40:00,377.000000,17.100000,19.500000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000
25%,2025-11-07 16:50:00,419.000000,20.100000,39.500000,0.00000,0.000000,0.000000,0.000000,0.0,1.000000
50%,2026-01-13 19:00:00,442.000000,21.800000,45.000000,0.00000,0.000000,0.000000,0.000000,0.0,1.000000
75%,2026-03-21 21:10:00,490.000000,24.000000,50.000000,1.00000,0.000000,0.000000,1.000000,0.0,2.000000
max,2026-05-27 23:20:00,2317.000000,29.900000,70.000000,3.00000,1.000000,40.000000,44.000000,0.0,5.000000
std,NaN,144.771006,2.761826,7.314519,0.91836,0.194571,3.236748,4.545337,0.0,0.977780


In [35]:
# 4.6 Export final Room 2.3 dataset


output_file = (
    "room_2_3_final.csv"
)


room_2_3_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    room_2_3_final.shape
)

Saved: room_2_3_final.csv
Final shape: (38645, 10)
